In [1]:
import os
import tomllib
os.chdir("/workspace")  # src is not a separately installed package yet.

from voc import collector as co
from src.preparation.dataset import COLUMNS, prepare_dataset

config = co.load_config("config/collector.toml")
raw = co.get_raw_dataset(config["raw_store"])
with open("config/preparation.toml", "rb") as stream:
    rules = tomllib.load(stream)
prepared = prepare_dataset(raw.data, rules)

assert list(prepared.data.columns) == COLUMNS
assert prepared.report["input_count"] == len(raw.data)
assert prepared.report["row_count"] == len(prepared.data)
assert prepared.report["row_count"] + prepared.report["excluded_count"] == len(raw.data)
print(prepared.data.shape)
print(prepared.report["counts_by_platform"])

(79713, 18)
{'google_play': 66579, 'x': 6553, 'app_store': 5253, 'youtube': 1328}


In [2]:
with_titles = prepare_dataset(raw.data, rules | {"combine_title_body": True})
assert with_titles.data["record_id"].equals(prepared.data["record_id"])
assert with_titles.data["text"].equals(prepared.data["text"])
changed = with_titles.data["model_text"].ne(prepared.data["model_text"])
print("Rows with changed model input:", int(changed.sum()))
print("Normalized dates:", prepared.report["date_normalizations"])

Rows with changed model input: 5253
Normalized dates: 72936


In [3]:
prepare_dataset(raw.data, {"unknown_rule": True})

ValueError: Unknown preparation rule

In [7]:
import os
from zenml.client import Client
run = Client().get_pipeline_run("dfc491f9-4ff3-4e53-8cf4-b464832bc65f")
assert run.status.value == "completed"
assert set(run.steps["raw_revision"].outputs) == {"raw_reference"}


In [8]:
client = Client()

first = client.get_pipeline_run("104f0f72-773e-47a7-8f9a-1aa251ccb9cb")
second = client.get_pipeline_run("dfc491f9-4ff3-4e53-8cf4-b464832bc65f")

first_artifact = first.steps["prepare"].outputs["workable"][0]
second_artifact = second.steps["prepare"].outputs["workable"][0]

assert first_artifact.id != second_artifact.id
assert first_artifact.load().equals(second_artifact.load())

print("Different artifact IDs, identical dataset contents.")

Different artifact IDs, identical dataset contents.
